# 5. Claude Code 연동 — AI와 함께 코딩하기

> **제5장** · 대응하는 이론편 내용: 없음(2권 고유)
> **예상 소요**: 60분 (설치 15분 + 실습 45분)
> **필요 사양**: 인터넷 연결, Anthropic 계정

---

## 이 장에서 하는 일

앞의 04장에서 **직접 만든 선형회귀 코드**를 재료로, AI 코딩 도구와 함께 개선해 본다.

| 절 | 하는 일 |
|---|---|
| 1 | 왜 4장 다음에 이걸 하는가 |
| 2 | Claude Code 설치 |
| 3 | `CLAUDE.md` 작성 — 프로젝트 맥락 알려주기 |
| 4 | 실습: 04번 코드를 개선하기 |
| 5 | **AI가 준 코드를 검증하는 법** ★ |
| 6 | 무엇을 맡기고 무엇을 직접 할 것인가 |

**5절이 이 장의 핵심이다.** AI가 코드를 빨리 만들어 주는 것보다,
그 코드가 맞는지 판단하는 능력이 더 중요하기 때문이다.

---

## 1. 왜 지금인가

AI 코딩 도구를 1장(환경 구축)에서 다루지 않고 여기까지 미룬 데는 이유가 있다.

**아직 시킬 것이 없었다.** 1~3장은 환경을 만들고 기본 도구를 익히는 단계라,
AI에게 부탁할 만한 "내 코드"가 없었다.

**판단할 기준이 없으면 위험하다.** 04장에서 우리는 선형회귀를 직접 구현하면서 다음을 겪었다.

- 그래디언트를 손으로 유도하고 수치 미분으로 검증했다
- 학습률을 잘못 잡아 발산하는 것을 봤다
- 표준화가 왜 필요한지 실패를 통해 이해했다
- 정규방정식과 대조해 결과가 맞는지 확인했다

**이 경험이 있어야 AI가 준 코드를 판단할 수 있다.** 만약 이 과정 없이 AI에게
"선형회귀 코드 짜줘"라고 했다면, 돌아온 코드가 왜 표준화를 하는지, 학습률을 왜 그 값으로 잡았는지
모른 채 복사해 쓰게 된다. 그러면 조금만 상황이 달라져도 손을 못 댄다.

> **이 책이 AI 도구를 대하는 태도**
>
> AI는 타자를 대신 쳐 주는 도구가 아니라, **내가 이해한 것을 더 빨리 구현하게 해 주는 도구**다.
> 이해하지 못한 것을 대신 이해해 주지는 않는다.

---

## 2. Claude Code 설치

Claude Code는 터미널에서 동작하는 AI 코딩 도구이고, VS Code 확장으로도 쓸 수 있다.

### 준비물

| 항목 | 확인 방법 |
|---|---|
| Node.js | `node --version` (없으면 nodejs.org에서 설치) |
| Anthropic 계정 | 유료 구독 필요 |

> **구독이 없어도 이 장을 읽는 의미는 있다.** 5절의 "AI가 준 코드를 검증하는 법"은
> 어떤 도구를 쓰든 똑같이 적용된다. 설치 확인 셀은 "미설치"로 나오지만 오류가 아니다.
>
> 또한 **이 구독은 25장에서 쓸 API 키와 다르다.** API 키는 별도로 발급받아야 한다.

### 설치 절차

**1) Node.js 설치** (이미 있으면 건너뛴다)

`https://nodejs.org` 에서 **LTS 버전**을 내려받아 설치한다. 기본 설정 그대로 진행하면 된다.

설치 후 **새 터미널**에서 확인한다.

```
node --version
npm --version
```

**2) Claude Code 설치**

```
npm install -g @anthropic-ai/claude-code
```

`-g`는 전역 설치를 뜻한다. 어느 폴더에서든 `claude` 명령을 쓸 수 있게 된다.

설치 확인:

```
claude --version
```

**3) 작업 폴더에서 실행**

```
cd C:\ai-practice
claude
```

처음 실행하면 브라우저가 열리며 로그인을 요청한다. Anthropic 계정으로 로그인하면 된다.

**4) VS Code 확장 (선택)**

VS Code 확장 탭(`Ctrl+Shift+X`)에서 **"Claude Code"**(게시자: Anthropic)를 검색해 설치한다.
터미널 대신 편집기 안에서 쓸 수 있고, 코드 변경 사항을 **차이 비교(diff) 화면으로 확인**할 수 있어 편리하다.

> 확장은 CLI를 감싼 것이므로, **CLI를 먼저 설치하고 로그인해 두어야** 한다.

### 설치 확인

아래 셀로 준비 상태를 점검한다.

In [ ]:
import subprocess
import shutil

print("=" * 55)
print("AI 코딩 도구 준비 상태")
print("=" * 55)

def check(cmd, name, install_hint):
    path = shutil.which(cmd)
    if path is None:
        print(f"[없음] {name}")
        print(f"       → {install_hint}")
        return False
    try:
        r = subprocess.run([cmd, "--version"], capture_output=True,
                           text=True, timeout=15)
        ver = (r.stdout or r.stderr).strip().split("\n")[0]
        print(f"[OK]   {name:<14}{ver}")
        return True
    except Exception as e:
        print(f"[확인 실패] {name}: {e}")
        return False

ok_node = check("node", "Node.js", "https://nodejs.org 에서 LTS 설치")
ok_npm  = check("npm",  "npm", "Node.js와 함께 설치됨")
ok_cc   = check("claude", "Claude Code", "npm install -g @anthropic-ai/claude-code")

print("-" * 55)
if ok_node and ok_cc:
    print("[준비 완료] 3절로 진행하세요.")
elif ok_node:
    print("Node.js는 있습니다. Claude Code를 설치하세요:")
    print("  npm install -g @anthropic-ai/claude-code")
else:
    print("Node.js부터 설치하세요. 설치 후 터미널을 새로 열어야 합니다.")

print()
print("※ 이 장의 나머지 셀은 Claude Code 없이도 실행됩니다.")
print("  3절의 CLAUDE.md 작성과 5절의 검증 방법은 그 자체로 유용합니다.")

---

## 3. CLAUDE.md — 프로젝트 맥락 알려주기

Claude Code는 대화를 시작할 때 프로젝트 루트의 **`CLAUDE.md`** 파일을 먼저 읽는다.
여기에 프로젝트의 규칙과 맥락을 적어 두면, 매번 같은 설명을 반복하지 않아도 된다.

### 무엇을 적을까

공식 안내에서 권하는 내용은 다음과 같다.

| 항목 | 예 |
|---|---|
| 프로젝트가 무엇인가 | "AI 교재 실습 코드" |
| 실행 방법 | "venv 활성화 후 실습 실행" |
| 코드 스타일 | "한글 주석, 변수명은 영문" |
| 지켜야 할 규칙 | "torch는 requirements.txt에 넣지 않는다" |

### 가장 중요한 원칙 — 짧게 쓴다

`CLAUDE.md`는 **모든 대화의 시작마다 읽힌다.** 즉 여기 적은 모든 줄이 매번 문맥을 차지한다.
이론편 20.4절에서 다룬 문맥 창을 떠올리면 이해가 쉽다 — 길어질수록 정작 중요한 지시가 묻힌다.

따라서 **코드를 통째로 붙여 넣지 말고, 파일 경로로 가리키는 편**이 낫다.

> `claude` 실행 후 `/init` 명령을 쓰면 프로젝트 구조를 훑어 초안을 만들어 준다.
> 그것을 손보는 방식이 편하다.

아래 셀은 이 책의 실습에 맞춘 `CLAUDE.md`를 생성한다.

In [ ]:
from pathlib import Path

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent

content = """# AI 교재 실습 프로젝트

## 프로젝트 개요
『이론으로 배우는 인공지능』(이론편)과 짝을 이루는 실습 코드 저장소.
이론편에서 손으로 계산한 값을 코드로 검증하는 것이 핵심 목적.

## 실행 환경
- Windows 11 / Python 3.13 / 가상환경 venv
- PyTorch는 CUDA 빌드 (--index-url 지정해 설치)
- GPU 8GB 기준으로 실습 규모를 잡음

## 폴더 구조
- part1/ ~ : 실습 파일(.ipynb). 파일명은 `NN_주제.ipynb`
- manuscript/ : 인쇄용 원고(.md)
- utils/ : 공통 함수 (gpu.py, seed.py)
- data/samples/ : 실습 데이터

## 코드 작성 규칙
- 주석과 출력 메시지는 한국어, 변수·함수명은 영문
- 실습 파일은 셀 단위로 부분 실행 가능해야 함
  → 각 절 첫 셀에 필요한 import를 다시 넣을 것
- 이론편 값과 대조하는 부분은 assert 로 검증
- 재현성이 필요하면 utils.seed.set_seed(42) 사용

## 하지 말아야 할 것
- requirements.txt 에 torch 를 추가하지 말 것
  (CUDA 버전 지정이 필요해 별도 설치해야 함)
- 데이터·모델 파일을 저장소에 커밋하지 말 것 (.gitignore 참조)
- GPU 8GB를 넘는 모델을 기본 예제로 쓰지 말 것

## 참고
- 이론편 원고: ../vol1/ (있는 경우)
- 각 실습 상단에 대응하는 이론편 절이 표시되어 있음
"""

path = root / "CLAUDE.md"
path.write_text(content, encoding="utf-8")

print(f"생성: {path}")
print(f"길이: {len(content)}자 / {len(content.splitlines())}줄")
print()
print("-" * 55)
print(content)

### 이 파일이 실제로 하는 일

`CLAUDE.md`가 있고 없고의 차이를 예로 들면 이렇다.

**없을 때** — "선형회귀 코드에 조기 종료를 추가해줘"라고 하면, AI는 이 프로젝트가 주피터 노트북 기반인지,
한글 주석을 쓰는지, 어떤 스타일을 따르는지 모른 채 일반적인 코드를 만든다.

**있을 때** — 위 규칙을 이미 읽었으므로 주피터 셀 형태로, 한글 주석을 달아, `assert` 검증까지 붙여 준다.

### 주의할 점

`CLAUDE.md`에 적은 것이 **항상 지켜지지는 않는다.** 지시가 많아질수록 일부가 무시될 수 있다.
따라서 정말 중요한 규칙은 짧게 몇 개만 적고, 그때그때 대화에서 다시 강조하는 편이 확실하다.

---

## 4. 실습 — 04번 코드를 개선하기

이제 실제로 써 본다. **04장에서 만든 선형회귀 코드**가 재료다.

### 실습 전에 — 요구사항을 먼저 말로 정리한다

AI에게 부탁하기 전에 **내가 무엇을 원하는지 스스로 정리하는 것**이 첫 단계다.
이것을 건너뛰면 애매한 결과가 나오고, 그것을 고치느라 더 오래 걸린다.

**나쁜 요청**

```
선형회귀 코드 좀 개선해줘
```

무엇을 개선할지 정하지 않았으므로, AI가 알아서 이것저것 바꾼다. 그중 무엇이 필요한지 판단하기 어렵다.

**좋은 요청**

```
utils/linreg.py 를 만들어줘.

04장의 train() 함수를 옮기되 다음을 추가한다:
1. 조기 종료 - 검증 손실이 patience 에폭 동안 개선되지 않으면 중단
2. 표준화를 클래스 안에서 자동 처리 (fit 에서 계산, predict 에서 적용)
3. 학습 이력을 dict 로 반환 (train_loss, val_loss)

제약:
- NumPy만 사용 (sklearn 금지)
- 주석은 한국어
- 이론편 5.4절 값과 대조하는 테스트 코드도 함께
```

**차이는 구체성이다.** 무엇을, 왜, 어떤 제약 아래에서 원하는지 밝혔다.
이렇게 쓰려면 **내가 먼저 이해하고 있어야 한다.**

### 실습 순서

터미널에서 다음과 같이 진행한다.

```
cd C:\ai-practice
claude
```

Claude Code가 실행되면 위의 "좋은 요청"을 그대로 입력한다.

**여기서 중요한 것** — AI가 만든 파일을 열어 **직접 읽어 보라.**
바로 실행하지 말고, 다음을 확인한다.

| 확인 항목 | 어떻게 |
|---|---|
| 그래디언트 식이 맞는가 | 04장에서 유도한 식과 비교 |
| 표준화를 어디서 하는가 | fit에서 통계를 구하고 predict에서 쓰는가 |
| 조기 종료 조건이 맞는가 | 검증 손실 기준인가, 훈련 손실 기준인가 |
| 재현성이 있는가 | 시드를 받는가 |

아래 5절에서 이 검증을 코드로 하는 방법을 다룬다.

---

## 5. AI가 준 코드를 검증하는 법 ★

이 장에서 가장 중요한 부분이다.

AI가 만든 코드는 **그럴듯해 보이지만 틀린 경우가 있다.** 문법 오류처럼 바로 드러나는 것이 아니라,
돌아가긴 하는데 결과가 미묘하게 다른 종류의 오류다. 이런 것은 눈으로 읽어서는 잘 안 보인다.

**해법은 우리가 이미 갖고 있다 — 이론편에서 손으로 구한 값이다.**

### 검증 방법 세 가지

| 방법 | 무엇을 확인 | 언제 쓰나 |
|---|---|---|
| 이론편 값 대조 | 결과가 맞는가 | 이론이 있는 부분 |
| 수치 미분 | 그래디언트 식이 맞는가 | 미분을 직접 구현할 때 |
| 극단 사례 | 예외 처리가 되는가 | 항상 |

아래는 검증용 함수를 미리 만들어 두는 예다. AI에게 코드를 받으면 이 함수들로 통과시켜 본다.

In [ ]:
import numpy as np


def verify_gradient(loss_fn, grad_fn, params, h=1e-5, tol=1e-4):
    """수치 미분으로 그래디언트 구현이 맞는지 확인한다 (04장 참조).

    loss_fn : params 를 받아 손실을 돌려주는 함수
    grad_fn : params 를 받아 그래디언트 배열을 돌려주는 함수
    """
    analytic = np.asarray(grad_fn(params), dtype=float)
    numeric = np.zeros_like(analytic)

    params = np.asarray(params, dtype=float)
    for i in range(len(params)):
        p_plus = params.copy();  p_plus[i]  += h
        p_minus = params.copy(); p_minus[i] -= h
        numeric[i] = (loss_fn(p_plus) - loss_fn(p_minus)) / (2 * h)

    diff = np.abs(analytic - numeric).max()
    passed = diff < tol

    print(f"  수식 유도 : {analytic.round(6)}")
    print(f"  수치 미분 : {numeric.round(6)}")
    print(f"  최대 차이 : {diff:.2e}  → {'통과' if passed else '실패'}")
    return passed


def verify_against_book(actual, expected, label, tol=1e-3):
    """이론편에서 손으로 구한 값과 대조한다."""
    actual = np.atleast_1d(np.asarray(actual, dtype=float))
    expected = np.atleast_1d(np.asarray(expected, dtype=float))
    passed = np.allclose(actual, expected, atol=tol)
    mark = "통과" if passed else "실패"
    print(f"  [{mark}] {label}")
    print(f"         계산값 {actual.round(4)}  /  이론편 값 {expected.round(4)}")
    return passed


print("검증 도구 준비 완료")
print("  verify_gradient()      - 수치 미분으로 그래디언트 확인")
print("  verify_against_book()  - 이론편 손계산 값과 대조")

### 실제로 검증해 보기

AI가 만들었다고 가정한 선형회귀 코드를 검증한다. 여기서는 예시로 코드를 직접 넣었지만,
실제로는 AI가 만든 파일을 불러와 같은 방식으로 확인하면 된다.

**일부러 미묘한 오류를 하나 심어 두었다.** 검증 도구가 그것을 잡아내는지 보자.

In [ ]:
import numpy as np

# ── AI가 만들었다고 가정한 코드 (오류가 하나 숨어 있다) ──
def loss_ai(params, X, y):
    w, b = params
    pred = w * X + b
    return np.mean((pred - y) ** 2)

def grad_ai(params, X, y):
    w, b = params
    pred = w * X + b
    error = pred - y
    dw = np.mean(error * X)      # ← 2가 빠져 있다
    db = 2 * np.mean(error)
    return np.array([dw, db])
# ────────────────────────────────────────────────

# 검증용 소규모 데이터
rng = np.random.RandomState(0)
X_v = rng.uniform(-2, 2, 30)
y_v = 1.5 * X_v + 0.5 + rng.randn(30) * 0.2
test_params = np.array([0.8, 0.2])

print("=" * 55)
print("AI 코드 검증 - 그래디언트")
print("=" * 55)
ok = verify_gradient(
    loss_fn=lambda p: loss_ai(p, X_v, y_v),
    grad_fn=lambda p: grad_ai(p, X_v, y_v),
    params=test_params,
)
print()
if not ok:
    print("→ 그래디언트 구현에 문제가 있다.")
    print("  첫 번째 값(dw)만 어긋났으므로 dw 계산을 살펴봐야 한다.")
    print("  04장에서 유도한 식과 비교해 보자:")
    print("    dL/dw = 2 * mean((pred - y) * X)")

In [ ]:
import numpy as np

# 문제를 고친 버전
def grad_fixed(params, X, y):
    w, b = params
    pred = w * X + b
    error = pred - y
    dw = 2 * np.mean(error * X)   # 2를 넣었다
    db = 2 * np.mean(error)
    return np.array([dw, db])

print("=" * 55)
print("수정 후 재검증")
print("=" * 55)
ok = verify_gradient(
    loss_fn=lambda p: loss_ai(p, X_v, y_v),
    grad_fn=lambda p: grad_fixed(p, X_v, y_v),
    params=test_params,
)
print()
print("[결론] 수치 미분 검증이 없었다면 이 오류를 놓쳤을 것이다.")
print()
print("실제로 이 오류가 있어도 학습은 '돌아간다'.")
print("그래디언트가 절반 크기라 학습률이 절반인 것과 같아지므로,")
print("느리게나마 수렴해서 문제를 알아채기 어렵다.")

### 왜 이런 오류가 위험한가

방금 본 오류는 **프로그램을 멈추지 않는다.** 그래디언트가 실제의 절반이라
학습률이 절반인 것과 같은 효과가 나므로, 조금 느릴 뿐 결국 수렴한다.

이런 종류의 오류는 다음과 같이 드러난다.

- "왜 이 모델은 학습률을 남들보다 크게 잡아야 하지?"
- "논문과 같은 설정인데 왜 결과가 다르지?"

원인을 찾기가 매우 어렵다. **그래서 처음부터 검증하는 것이 훨씬 싸다.**

12장에서 역전파를 직접 구현할 때 이 검증 도구를 다시 쓴다.
층이 여러 개면 오류가 숨을 곳이 더 많아지기 때문이다.

---

## 6. 무엇을 맡기고 무엇을 직접 할 것인가

### 맡기기 좋은 것

| 종류 | 예 | 이유 |
|---|---|---|
| 반복적인 작업 | 여러 설정으로 실험 돌리는 반복문 | 실수가 잦고 지루함 |
| 시각화 코드 | 그래프 여러 개 배치, 색·라벨 조정 | 결과를 보면 맞는지 바로 앎 |
| 문서화 | docstring, 주석, README | 검토가 쉬움 |
| 리팩터링 | 함수로 분리, 이름 정리 | 동작이 같은지 테스트로 확인 가능 |
| 익숙지 않은 라이브러리 문법 | "이걸 pandas로 어떻게 쓰지" | 문서 찾는 시간 절약 |

### 직접 해야 하는 것

| 종류 | 왜 |
|---|---|
| **핵심 알고리즘의 이해** | 이해 없이 받으면 고칠 수 없다 |
| **검증 기준 정하기** | 무엇이 맞는지는 사람이 정한다 |
| **결과 해석** | 숫자가 무엇을 뜻하는지 판단 |
| **왜 이 방법인지 결정** | 설계는 사람의 몫 |

### 이 책에서의 방침

**직접 구현하는 장**(04, 09, 10, 12, 13, 18장)은 AI에게 맡기지 않기를 권한다.
이 실습들은 "돌아가는 코드"를 얻는 것이 목적이 아니라 **원리를 손으로 확인하는 것**이 목적이기 때문이다.

반면 **응용 장**(18, 24, 26번 등)에서는 적극적으로 활용해도 좋다.
그때쯤이면 결과가 맞는지 판단할 눈이 생겨 있을 것이다.

---

## 7. 정리

### 이 장의 핵심

| 항목 | 요점 |
|---|---|
| 순서 | **직접 구현 → 이해 → AI 활용**. 반대는 위험 |
| 요청하기 | 무엇을·왜·어떤 제약인지 구체적으로 |
| CLAUDE.md | 짧게. 매 대화마다 읽히므로 문맥을 차지함 |
| **검증** | 이론편 값 대조 + 수치 미분 + 극단 사례 |
| 맡길 것 | 반복 작업, 시각화, 문서화, 리팩터링 |
| 직접 할 것 | 핵심 알고리즘 이해, 검증 기준, 결과 해석 |

### 만든 것

- `CLAUDE.md` — 프로젝트 맥락 파일
- `verify_gradient()` — 수치 미분 검증
- `verify_against_book()` — 이론편 값 대조

뒤의 두 함수는 앞으로 계속 쓴다.

### Part 1을 마치며

여기까지가 1부다. 되짚어 보면 다음을 했다.

| 장 | 한 일 |
|---|---|
| 01 | 환경 구축 |
| 02 | NumPy로 이론편 4장 값 검증 |
| 03 | 시각화 — PCA·손실 곡선 |
| 04 | 선형회귀 직접 구현, 표준화의 필요성 |
| 05 | AI 도구 활용과 검증 |

### 다음 장

**6. 데이터 전처리와 탐색** — 2부가 시작된다.
04장에서 직접 만든 선형회귀를 라이브러리로 다시 해 보고, 분류 문제로 넘어간다.